### Dialogue to deeal with the admin routes for my blog site.  This includes editing and saving blog posts, though not as yet creating from scratch

Features still to be added
1. buttons on the relevant blog pages to allow edit - Done
2. buttons on the relevant blog pages to allow delete - Done
3. buttons on the relevant blog pages to allow download - Done
4. Add the capability to change tags for a post _still to do


In [ ]:
#| default_exp admin_v1

In [ ]:
#| hide
from nbdev.showdoc import *
import nbdev

In [ ]:
#| export
from fastlite import Database
from pathlib import Path
from datetime import datetime, timedelta
import my_blog.config as config
from urllib.parse import quote, unquote
from fasthtml.common import *
from monsterui.all import *
from fasthtml_auth import AuthManager
from fasthtml.jupyter import *
import re
import frontmatter

In [ ]:
#| export
from my_blog.core_v5 import (layout, render_md, EnhancedRenderer, process_obsidian_images, process_strava_embeddings, 
    navbar, footer, hx_link, hx_attrs)

In [ ]:
#| export
# Route collection for deferred registration
_admin_routes = []

def route(path=None):
    """Decorator to collect routes without registering them immediately.
    Use @route('/path') or @route() for function-name-based paths."""
    def decorator(f):
        _admin_routes.append((path, f))
        return f
    if callable(path):  # @route without parens
        f, path = path, None
        _admin_routes.append((path, f))
        return f
    return decorator

In [ ]:
#| export
_state = None

In [ ]:
#| export
def register_admin_routes(app, state):
    """Register all collected routes with the app and setting state for the module"""
    global _state
    _state = state
    for path, handler in _admin_routes:
        if path:
            app.route(path)(handler)
        else:
            app.route(handler)

In [ ]:
#| export
def require_admin(req):
    user = req.scope.get('user')
    if not user or user.role != 'admin':
        return Response("Forbidden", status_code=403)


In [ ]:
#| export
@route('/admin/edit/{slug}')
def post_edit(req, htmx, slug: str):
    # Load post content from database
    if err := require_admin(req): return err
    p = load_post(slug)
    if not post:
        return layout(req, H2("Not Found"), P("Post not found."), title="Not Found", htmx=htmx)
    content = p['content']

    # Create edit page components and layout including textarea, output div, buttons for cancel, full_preview, save, save and return
    # main content area has id: 'main-content'
    frm = Form(
        Div(id='notify'),
        Grid(cls='grid grid-cols-2 gap-4')(
        Textarea(content, name='raw_post', id='raw_post', hx_post= f'/admin/post_simple_render', hx_target='#preview', hx_trigger='load, keyup changed delay:500ms',
        cls='w-full min-h-[80vh] text-lg overflow-hidden'),
        Div(id='preview', cls='text-lg border rounded p-4')
        ),
        Div(
            # Button('Cancel', cls=[ButtonT.secondary, ButtonT.sm], href=f'/blog/{slug}'),
            A('Cancel', href=f'/blog/{slug}', cls=[ButtonT.secondary, ButtonT.sm]),
            Button('Save', hx_post=f'/admin/save/{slug}', hx_target='#notify', hx_swap='innerHTML', cls=[ButtonT.primary, ButtonT.sm]),
            Button('Full Preview', cls=[ButtonT.primary, ButtonT.sm], hx_post=f'/admin/post_simple_render?full_render=True', 
            hx_target='#preview'),
            Button('Save & return', hx_post=f'/admin/save/{slug}?return_to_blog=true', hx_target='#notify', hx_swap='innerHTML', cls=[ButtonT.primary, ButtonT.sm]),           
            id='button_bar'
        ),
        Hidden(name='slug', value=slug)
    )
    return edit_layout(req, H1(p['title'], cls="text-3xl font-bold mb-2"), Span(p['created'].strftime('%B %d, %Y'), cls="text-muted-foreground text-sm mb-8 block"), 
        frm, title=p['title'], htmx=htmx)

    # 

In [ ]:
#| export
@route('/admin/save/{slug}')
def save_post(htmx, slug: str, raw_post: str, return_to_blog: bool = False):
    p = load_post(slug)
    post_id = p['id']
    content = raw_post
    try: 
        _state.posts_t.update(dict(id=post_id, content=content, updated=datetime.now()))
    except Exception as inst:
        return Toast(f"Unable to save post: {inst}", alert_cls=AlertT.error)
    if return_to_blog:
        return HtmxResponseHeaders(redirect=f'/blog/{slug}')
    else:
        return Toast("Post saved :)", alert_cls=AlertT.info)


The route admin/edit{slug} works well, I now need a route and function to delete a post based upon the slug and also one to download the the post content, title, tags, excerpt to a local markdown file with Obsidian format. Can you provide two functions to do that using the same approach as above

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here are the two functions:

```python
@route('/admin/delete/{slug}')
def delete_post(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    return HtmxResponseHeaders(redirect='/blog')
```

```python
@route('/admin/download/{slug}')
def download_post(slug: str):
    p = load_post(slug)
    if not p: return Response("Post not found", status_code=404)
    fm = dict(title=p['title'], tags=p.get('tags', []), excerpt=p.get('excerpt', ''), date=p['created'].strftime('%Y-%m-%d'))
    md = f"---\n{'\n'.join(f'{k}: {v}' for k,v in fm.items())}\n---\n\n{p['content']}"
    fname = f"{slug}.md"
    return Response(md, media_type='text/markdown', headers={'Content-Disposition': f'attachment; filename="{fname}"'})
```

A couple of notes on these:

- **Delete** removes the post and redirects back to `/blog`. You might want to add a confirmation step (e.g. a modal) before calling this route — happy to help with that next.
- **Download** builds YAML frontmatter in Obsidian format with title, tags, excerpt, and date, then returns it as an `.md` file attachment. I used `dict` + a simple join for the frontmatter rather than the `frontmatter` library since the structure is straightforward — but the tags field will render as a Python list string rather than proper YAML list syntax. If you'd prefer to use the `frontmatter` library for cleaner YAML output, let me know and I'll revise.

If the above is called from the blog page with the following: Button("Delete", hx_post=f"/admin/delete/{slug}", hx_confirm="Delete this post?", cls="uk-btn uk-btn-default uk-btn-xs"), then what will the Redirect cause to happen and how does this work with htmx?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question — there's a subtle but important interaction here.

When the user clicks that button, here's the flow:

1. `hx_confirm` fires first — the browser shows a native confirm dialog ("Delete this post?"). If the user cancels, nothing happens.
2. If confirmed, HTMX sends an **AJAX POST** to `/admin/delete/{slug}`.
3. Your handler deletes the post and returns `HtmxResponseHeaders(redirect='/blog')`, which sets the `HX-Redirect` response header.
4. HTMX sees the `HX-Redirect` header and performs a **full page navigation** to `/blog` — not an AJAX swap. It essentially does `window.location.href = '/blog'`.

The key distinction is between `HX-Redirect` and a normal HTTP 301/302 redirect:
- A **normal HTTP redirect** on an HTMX request would be followed transparently by the browser's fetch — HTMX would get the redirected page content and try to swap it into the target element, which is usually not what you want.
- **`HX-Redirect`** tells HTMX to stop the swap and do a full browser navigation instead, which is exactly right here since the current page (the deleted post) no longer exists.

So the user experience is: click Delete → confirm → the post disappears and they land on the blog index. Clean and simple.

One thing to note: since there's no `hx_target` on your button, HTMX would default to swapping the button itself — but that never happens because `HX-Redirect` takes priority and triggers navigation before any swap occurs.

If this was coming from a page with just the one post then that is fine but if it is coming from the main blog page with a list of posts, and this one has been chosen for deletion, then its only the main-content div that needs to be refreshed, hence should i not set a hx_target and target refreshing that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're absolutely right. If the user is on the blog listing page, redirecting to `/blog` triggers a full page reload when all you really need is to refresh the post list. Much smoother to just swap the content.

Two options:

**Option A** — Change the button to target `#main-content` and have the handler return the refreshed listing:

```python
Button("Delete", hx_post=f"/admin/delete/{slug}", hx_confirm="Delete this post?", hx_target="#main-content", hx_swap="innerHTML", cls="uk-btn uk-btn-default uk-btn-xs")
```

Then update the handler to return the blog listing content instead of a redirect:

```python
@route('/admin/delete/{slug}')
def delete_post(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    return render_blog_list()
```

**Option B** — If you want the button to work from *both* the single post page (where redirect is appropriate) and the listing page (where a swap is better), you could check `htmx` to see if it's an HTMX request with a target, and branch accordingly.

Which scenario do you need to support — listing page only, or both contexts?

I would prefer to do it the second way as I am keen to better understand how to deal with different htmx scenarios

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@route('/admin/delete/{slug}')
def delete_post(htmx, slug: str):
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    if htmx and htmx.target == 'main-content': return render_blog_list()
    return HtmxResponseHeaders(redirect='/blog')
```

The key line is `htmx.target` — HTMX sends the `HX-Target` header with every request, telling the server which element will receive the swapped content. So:

- From the **listing page**, where your button has `hx_target="#main-content"`, the header arrives as `main-content` (without the `#`). The handler returns the fresh listing HTML and HTMX swaps it in — no full page reload.
- From the **single post page**, where there's no `hx_target`, `htmx.target` will be `None` (or the button's own id), so it falls through to the redirect, which navigates the user away from the now-deleted post.

For the listing page button, make sure you include the target:

```python
Button("Delete", hx_post=f"/admin/delete/{slug}", hx_confirm="Delete this post?", hx_target="#main-content", hx_swap="innerHTML", cls="uk-btn uk-btn-default uk-btn-xs")
```

And for the single post page, the simpler version without `hx_target` will trigger the redirect path.

In [ ]:
#| export
@route('/admin/delete/{slug}')
def delete_post(req,htmx, slug: str):
    if err := require_admin(req): return err
    p = load_post(slug)
    if not p: return Toast("Post not found", alert_cls=AlertT.error)
    try: _state.posts_t.delete(p['id'])
    except Exception as e: return Toast(f"Unable to delete: {e}", alert_cls=AlertT.error)
    return HtmxResponseHeaders(redirect='/blog')

In [ ]:
#| export
@route('/admin/download/{slug}')
def download_post(req, slug: str):
    if err := require_admin(req): return err
    p = load_post(slug)
    if not p: return Response("Post not found", status_code=404)
    tag_rows = _state.pdb.q("""SELECT t.name FROM tags t JOIN post_tags pt ON t.id = pt.tag_id WHERE pt.post_id = ?""", [p['id']])
    tags = [r['name'] for r in tag_rows]
    fm = dict(title=p['title'], tags=tags, excerpt=p.get('excerpt', ''), created=str(p['created']))
    fm['private'] = bool(p.get('private', False))
    if p.get('updated'): fm['updated'] = str(p['updated'])
    post = frontmatter.Post(p['content'], **fm)
    md = frontmatter.dumps(post)
    fname = f"{p['title']}.md"
    return Response(md, media_type='text/markdown', headers={'Content-Disposition': f'attachment; filename="{fname}"'})

In the above routes /admin/download/{slug}, delete_post(...), post_edit() I am concerned that if somebody on the web wanted to hack the site they could use the above routes to access or delete posts. Also some posts are private and should not be accessible (the main access points are protected).  fasthtml_auth provides a decorator @auth.require_admin() but I am not sure how to use this with the way we are registering routes using the register_admin_routes methodology.  We could also check for the user role using the req get user and check their role as an alternative.  Do you agree this is a possible security issue and if so which of the above routes would you recommend to block it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, this is a genuine security concern — all three routes should be protected since they expose read, write, and delete operations on potentially private content.

On the two approaches:

- **`@auth.require_admin()`** — clean and declarative, but you're right that it may not play well with the deferred registration pattern, since the decorator runs at definition time before the auth system may be fully wired up.
- **Checking the user role via `req`** — more flexible and works naturally with your registration approach. Something like:

```python
if not req.scope.get('user') or req.scope['user'].role != 'admin':
    return Response("Forbidden", status_code=403)
```

This can be wrapped into a small helper function (e.g. `check_admin(req)`) and called at the top of each protected route.

As for which routes to protect — which of the three do you think need it, and why? Have a think and I'll confirm or challenge your reasoning!

<details class='token-usage-details'><summary>$0.1897</summary>

`total=33,005 | in=32,719 | out=286 | cached=0.0% | cache_new=32,066 | $0.1897`

</details>


At present when using the download_post function I am setting the filename to the post title with .md.  This is the way the post is initially created in Obscidian.  Should I also save the slug and load it on subsequent uploads, since if somebody changes the title then when it creates a new slug all of the images will be lost etc?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a really important consideration! Yes, storing the slug in the frontmatter is a good defensive move. The slug is typically derived from the original title at creation time, and if someone edits the title later, you'd want to preserve the original slug to maintain all the image links and internal references.

So in `download_post`, adding `slug` to the frontmatter dict is simple:

```python
fm = dict(title=p['title'], slug=slug, tags=tags, ...)
```

But the more interesting question is the **upload/re-import side** — when someone uploads a previously downloaded file, does your import process currently read frontmatter fields, and would it honour a `slug` field if present?

How does your upload/import currently work?

Yes it does and so if a slug is present I can make it use that rather than recreate it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's exactly the right approach. So the logic on upload would be:

- If `slug` is present in frontmatter → use it as-is
- If not → generate from the title as before

Before you implement that, it's worth thinking about one edge case: what if someone uploads a file with a slug that **already exists** in the database (e.g. they're re-importing an edited version of an existing post)? Should it update the existing post, or create a new one?

It should probably check if overwriting the existing post is what is required. I can use a hx_confirm to do this I think. The upload function is in core_v5 with the function process_upload. Some suggestion as to how to implement this would be useful

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good thinking. Before suggesting an implementation, it's worth clarifying the flow a little:

When the user uploads a file, at what point would you want to show the confirm dialog? The challenge with `hx_confirm` is that it fires *before* the request is sent — but you won't know if the slug already exists until the server has checked the database.

So you have two broad options:

1. **Server-side check first** — `process_upload` checks if the slug exists and returns a response asking the user to confirm overwrite (e.g. a modal or a confirmation button), which then triggers a second request with an `overwrite=True` flag.
2. **Two-step upload endpoint** — split into a "check" endpoint and a "confirm write" endpoint.

Option 1 is probably simpler and fits naturally with HTMX. Would you like to go with that approach?

I can check for confirm in the route but I am not sure of the mechanism for generating the confirmation dialogue. Please give a consise example

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In your upload route, when you get `'confirm'` back, return a small form that re-submits with `overwrite=True`:

```python
if status == 'confirm':
    return Div(
        P(message),
        Button("Yes, overwrite", hx_post=f"/admin/upload?overwrite=true", hx_vals=f'{{"slug":"{slug}"}}', 
               hx_target="#upload-result", hx_swap="innerHTML", cls=ButtonT.danger),
        Button("Cancel", hx_on_click="this.closest('#upload-result').innerHTML=''", cls=ButtonT.secondary),
        id="upload-result"
    )
```

This renders inline where your upload result normally appears. The "Yes" button re-POSTs to the same upload route, passing `overwrite=true` as a query param and the slug as an hx-val.

Your route would then need to accept `overwrite: bool = False` and pass it through to `process_upload`. Want me to show that?

<details class='token-usage-details'><summary>$0.0270</summary>

`total=41,334 | in=41,045 | out=289 | cached=98.9% | cache_new=204 | $0.0270`

</details>


So I don't have a current id of upload-result, its only created by the dialogue, is that ok? I don't understand the hx_vals structure, please explain.  The current /admin/upload route definition is: 
@route('/admin/upload')    
def post(upload2: list[UploadFile], overwrite:bool=False):
So I am not sure that the button to overwrite call is correct?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Three questions, let me take them one at a time:

**`id="upload-result"`** — yes that's fine. You're creating the div dynamically and giving it an id so the cancel button can find and clear it. It doesn't need to pre-exist.

**`hx_vals`** — this injects extra values into the HTMX request as if they were form fields. So `hx_vals='{"slug":"my-post"}'` adds `slug=my-post` to the POST body. It's a way of passing extra data without hidden form fields.

**The route** — you're right, the overwrite button can't call `/admin/upload` the same way since that expects a list of `UploadFile`. You have two options:

1. Add a separate `/admin/upload/confirm-overwrite` route that just handles the overwrite case, taking `slug`, `content` (stored temporarily) and `overwrite=True`
2. Store the pending upload content in session or a temp file, then the confirm button just passes the slug back

Which feels more natural to you?

<details class='token-usage-details'><summary>$0.0832</summary>

`total=41,695 | in=41,445 | out=250 | cached=72.4% | cache_new=11,140 | $0.0832`

</details>


If I go down the first route then how will it then resume to doenload the other associated files?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point — it won't easily, since the original file list is gone once the first request completes.

That makes option 2 (storing the pending content temporarily) the more practical choice. A simple approach would be to save the `.md` content to a temp file keyed by slug, then the confirm button just passes the slug back to a lightweight confirm route that reads from the temp file and completes the upload.

Does that sound workable to you?

<details class='token-usage-details'><summary>$0.0241</summary>

`total=41,873 | in=41,773 | out=100 | cached=98.5% | cache_new=402 | $0.0241`

</details>


Yes, pln you please outline the functions

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the outline:

1. **`save_pending(slug, content)`** — saves the raw md bytes to e.g. `/tmp/pending_{slug}.md`

2. **`load_pending(slug)`** — reads it back and returns the bytes

3. **A new `/admin/upload/overwrite` route** — accepts just `slug: str`, calls `load_pending(slug)`, then calls `process_upload(..., overwrite=True)`, and cleans up the temp file

The existing upload route would call `save_pending` before returning the confirm dialog.

Shall I show the code for any of these?

<details class='token-usage-details'><summary>$0.0837</summary>

`total=42,083 | in=41,937 | out=146 | cached=71.5% | cache_new=11,720 | $0.0837`

</details>


Can you outline the flow please, I am a bit confused by the sequence. Lets start with the call to /admin/upload with the file list. Please be concise

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the sequence:

1. **`/admin/upload`** receives the file list, processes images normally, and for the `.md` file calls `process_upload(...)`
2. **`process_upload`** checks if the slug exists:
   - **No** → saves normally, returns `True, "Post saved", slug`
   - **Yes** → calls `save_pending(slug, content)`, returns `'confirm', "Post already exists...", slug`
3. **`/admin/upload`** sees `'confirm'` → returns the confirmation dialog to the user (images already saved at this point)
4. User clicks **"Yes, overwrite"** → POSTs to `/admin/upload/overwrite` with just the slug
5. **`/admin/upload/overwrite`** calls `load_pending(slug)`, calls `process_upload(..., overwrite=True)`, deletes temp file, returns success message

Does that flow make sense now?

<details class='token-usage-details'><summary>$0.0868</summary>

`total=42,400 | in=42,176 | out=224 | cached=71.1% | cache_new=11,930 | $0.0868`

</details>


No because the images are not saved first, the md file is, and it only makes sense to save the images if we know that the .md file is going to be saved

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're absolutely right — the `.md` file determines whether the post exists, so the flow should be:

1. **`/admin/upload`** receives the file list, processes the `.md` file **first** via `process_upload()`
2. If `'confirm'` → save pending content, stash the image files too (or their bytes), return confirmation dialog. **No images saved yet.**
3. If `True` → post saved, **now** process and save the images
4. On overwrite confirm → `/admin/upload/overwrite` saves the post, **then** saves the stashed images

So you'll need to stash the image bytes alongside the md content. Does it make sense to extend `save_pending` to store both the md and the image files together?

<details class='token-usage-details'><summary>$0.4049</summary>

`total=42,705 | in=42,527 | out=178 | cached=0.0% | cache_new=42,279 | $0.4049`

</details>


In [ ]:
#| export
def replace_iframe(content: str):
    """ Comments re regex: The [^>] means not >, and so adding the * means everything up to the >.  The ? .*?< makes the .* non-greedy (lazy). Without it, if there were two iframes in the content, .* would greedily match from the first <iframe> all the way to the last </iframe>, swallowing everything in between — including any content between the two iframes. With .*?, it stops at the nearest </iframe>."""

    pattern = r'<iframe[^>]*>.*?</iframe>'

    content = re.sub(pattern, '<div class="border rounded p-4 text-center text-muted-foreground">iframe embed (full preview to see)</div>', content, flags=re.DOTALL)
    return content

In [ ]:
#| export
def replace_strava(content: str):
    """ The strava pattern is: {{strava:16611889793}}
    """
    pattern = r'\{\{strava:\d+\}\}'

    content = re.sub(pattern, '<div class="border rounded p-4 text-center text-muted-foreground">🚴 Strava embed (full preview to see)</div>', content, flags=re.DOTALL)
    return content

In [ ]:
test_strava = 'This is dummy text with a strava embed {{strava:1234567}} here'
expected = 'This is dummy text with a strava embed <div class="border rounded p-4 text-center text-muted-foreground">🚴 Strava embed (full preview to see)</div> here'
assert replace_strava(test_strava)==expected

In [ ]:
test_iframe_content = 'This is dummy text with an iframe <iframe src="https://www.google.com/maps/d/embed?mid=1NXlWhw7ZUcEQ_hGaXVAuuPz4DnGN_wQ&hl=en&ehbc=2E312F" width="640" height="480"></iframe> embedded'
expected = 'This is dummy text with an iframe <div class="border rounded p-4 text-center text-muted-foreground">iframe embed (full preview to see)</div> embedded'
assert replace_iframe(test_iframe_content) == expected

In [ ]:
#| export
@route('/admin/post_simple_render')
def post(raw_post: str,  slug: str, full_render: bool=False):
    p = load_post(slug)
    # Note - I think having the title and created in the form as hidden fields will be better than loading the post or this might slow things down
    if not full_render:
        # remove iframes and strava embeddings
        raw_post = replace_iframe(raw_post)
        # Remove Strava embeddings
        raw_post =replace_strava(raw_post)
    rendered = render_md(raw_post, renderer=EnhancedRenderer)
    if full_render:
        image_base = f"/static/image/post_images/{slug}"
        rendered = process_obsidian_images(rendered, image_base=image_base)
        rendered = process_strava_embeddings(rendered)
    if full_render:
        return Div(rendered, Script(src="https://strava-embeds.com/embed.js"))
    else:
        return Div(rendered)

In [ ]:
#| export
def load_post(slug):
    row = _state.posts_t.rows_where("slug = ?", [slug], limit=1)
    p = next((dict(r) for r in row), None)
    if not p: return None
    p['created'] = datetime.fromisoformat(p['created']) if isinstance(p['created'], str) else p['created']
    return p

In [ ]:
#| export
def edit_layout(req, *content, htmx, title=None):
    if htmx and htmx.request: return (Title(title), *content)
    main = Main(*content, cls='w-full max-w-full px-8 mx-auto py-8 space-y-8', id="main-content")
    return Title(title), Div(Div(navbar(req), cls='max-w-full px-8 mx-auto mt-4'), main, footer(), cls="flex flex-col min-h-screen")

### Export

In [ ]:
nbdev.nbdev_export()